# 🇩🇪 LFM3 für Home Assistant – deutsches Tool-Calling

Fine-Tune von [`LiquidAI/LFM3-Omni-2B`](https://huggingface.co/LiquidAI/LFM3-Omni-2B)
auf deutschen Home-Assistant-Befehlen: Befehl rein, Werkzeugaufruf raus.

```
„Mach das Licht im Wohnzimmer an."
    → control(service="light.turn_on", entity_id=["light.wohnzimmer_decke"])
```

**Vorbereitung**

1. Laufzeit → GPU wählen (T4 = Smoke-Test, L4/A100 = echter Lauf).
2. Links das 🔑-Symbol → Secret `HF_TOKEN` anlegen
   ([Token mit *write*](https://huggingface.co/settings/tokens)).
3. Alle Zellen von oben nach unten ausführen.

Repo: <https://github.com/Tyrell04/lfm3-ha-de> · Doku: `docs/TRAINING.md`

## 1 · Hardware prüfen

In [ ]:
import subprocess

import torch

print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip() or "keine GPU gefunden")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} · {props.total_memory / 1e9:.1f} GB · bf16: {torch.cuda.is_bf16_supported()}")
else:
    print("⚠️  Keine CUDA-GPU. Laufzeit → GPU umstellen, sonst dauert das Training ewig.")

## 2 · Installation

Pinnt `transformers==5.20.2` – LFM3-Omni braucht genau diese Version
(eigener Modellcode über `trust_remote_code`).

In [ ]:
%%capture
!pip install -q "transformers==5.20.2" peft accelerate datasets huggingface_hub pyyaml tqdm
# Colab bringt torchao 0.10 mit; peft >= 0.21 verlangt > 0.16 und bricht sonst ab.
!pip install -q -U "torchao>=0.16"
! [ -d lfm3-ha-de ] || git clone -q https://github.com/Tyrell04/lfm3-ha-de.git
%cd lfm3-ha-de
!git pull -q --ff-only || echo "kein Update (lokale Änderungen)"
!pip install -q -e . --no-deps
print("ok")

In [ ]:
import datasets
import peft
import transformers

print("transformers", transformers.__version__)
print("peft", peft.__version__)
print("datasets", datasets.__version__)

## 3 · Hugging Face Login

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN gesetzt:", os.environ["HF_TOKEN"][:8] + "…")
except Exception:
    print("Kein HF_TOKEN gefunden – Training läuft, der Upload am Ende wird übersprungen.")

## 4 · Konfiguration

`SMOKE = True` trainiert in wenigen Minuten auf einem Bruchteil der Daten –
gut, um die Pipeline zu prüfen. Für das echte Modell auf `False` stellen.

In [ ]:
BASE_MODEL = "LiquidAI/LFM3-Omni-2B"   # alternativ: "LiquidAI/LFM3-1.7B" (nur Text, kleiner)
SMOKE = True
EPOCHS = 1 if SMOKE else 2
OUTPUT_DIR = "outputs/lfm3-ha-de"
MODEL_REPO = "marc04/lfm3-ha-de"                     # Ziel auf Hugging Face
DATA_REPO = "marc04/homeassistant-tool-calling-lfm3-de"
PUSH_MODEL = False                                   # erst nach Review auf True

print(f"{BASE_MODEL} · {EPOCHS} Epoche(n) · smoke={SMOKE}")

## 5 · Daten

Zwei Quellen werden zu einem LFM3-Chatdatensatz verschmolzen:

* `mkenfenheuer/homeassistant-tool-calling` (`locale == "de"`) – Aufgabe
* `OpenAssistant/oasst2` (deutsche Threads, ~4 %) – Sprachgefühl

Das frühere `homeassistant-tool-calling-de` ist im mehrsprachigen Datensatz
aufgegangen, deshalb der Locale-Filter.

In [ ]:
LIMIT = " --limit 400 --oasst-limit 120" if SMOKE else ""
!python -m lfm3ha.cli prepare --out-dir data/lfm3-ha-de{LIMIT}

## 6 · Training

In [ ]:
import torch

vram = torch.cuda.get_device_properties(0).total_memory / 1e9 if torch.cuda.is_available() else 0
if vram >= 35:
    batch, accum = 8, 4
elif vram >= 20:
    batch, accum = 4, 8
else:
    batch, accum = 1, 16          # T4: 16 GB, kein bf16

rows = " --max-train-rows 400" if SMOKE else ""
print(f"{vram:.0f} GB VRAM → batch {batch} × accum {accum}")

In [ ]:
!python -m lfm3ha.cli train \
    --base-model {BASE_MODEL} \
    --dataset data/lfm3-ha-de \
    --output-dir {OUTPUT_DIR} \
    --epochs {EPOCHS} \
    --batch-size {batch} --grad-accum {accum}{rows}

## 7 · Auswertung

Drei getrennte Metriken: **parse** (gültiger `<tool_call>`), **function**
(richtige Namen), **exact** (Namen *und* Argumente). Ein Vergleich gegen das
Basismodell zeigt, ob das Training etwas gebracht hat.

In [ ]:
!python -m lfm3ha.cli eval \
    --base-model {BASE_MODEL} --dataset data/lfm3-ha-de \
    --limit 100 --out outputs/metrics_base.json

In [ ]:
!python -m lfm3ha.cli eval \
    --base-model {BASE_MODEL} --adapter {OUTPUT_DIR} --dataset data/lfm3-ha-de \
    --limit 100 --out outputs/metrics_lora.json

In [ ]:
import json
from pathlib import Path

fehlend = [p for p in ("outputs/metrics_base.json", "outputs/metrics_lora.json")
           if not Path(p).exists()]
if fehlend:
    raise SystemExit(
        f"{', '.join(fehlend)} fehlt – die beiden eval-Zellen darüber sind "
        "nicht durchgelaufen. Scrolle hoch und lies dort den Fehler."
    )

with open("outputs/metrics_base.json", encoding="utf-8") as f:
    base = json.load(f)
with open("outputs/metrics_lora.json", encoding="utf-8") as f:
    lora = json.load(f)

print(f"{'Metrik':<12}{'Basismodell':>14}{'+ LoRA':>12}")
for key in ("parse", "function", "exact"):
    before = base["tool_calls"][key]
    after = lora["tool_calls"][key]
    print(f"{key:<12}{before if before is not None else '-':>14}{after if after is not None else '-':>12}")
print(f"\nWerkzeug-Rows: {lora['tool_calls']['n']}")

## 8 · Demo

Vier deutsche Befehle im Beispielhaus. Prüfe, ob `service` und `entity_id`
zum Prompt passen – das ist die eigentliche Qualitätsfrage.

In [ ]:
!python -m lfm3ha.cli chat --base-model {BASE_MODEL} --adapter {OUTPUT_DIR}

## 9 · Veröffentlichen

Verschmilzt den Adapter mit dem Basismodell und lädt zu Hugging Face.
Auf einer T4 vorher **Laufzeit → Laufzeittyp ändern → High-RAM** wählen
(verschmelzen braucht ~10 GB CPU-RAM), oder `--adapter-only` nutzen.

In [ ]:
if PUSH_MODEL:
    !python -m lfm3ha.cli push \
        --adapter {OUTPUT_DIR} \
        --base-model {BASE_MODEL} \
        --repo-id {MODEL_REPO} \
        --dataset {DATA_REPO} \
        --metrics outputs/metrics_lora.json
else:
    print("PUSH_MODEL = False – nichts hochgeladen.")

## 10 · Nächste Schritte

* **Datensatz veröffentlichen:** `prepare` einmal mit
  `--push-to-hub marc04/homeassistant-tool-calling-lfm3-de` ausführen.
* **Mehr Qualität:** `--chat-ratio` senken (weniger OASST2 → strengeres
  Format), mehr Epochen erst nach Blick auf `exact`.
* **Sprache statt Text:** auf Audio trainieren – der Collator steht im
  offiziellen Cookbook-Notebook, der Rest dieses Repos bleibt gleich.
* **In Home Assistant testen:** der Agent braucht die Geräteliste im
  Systemprompt (`build_system_prompt`) und `temperature=0`.